# 3.4 · Imputación de huecos con modelos

**Tiempo estimado:** 30 min.

**Objetivos.**

1. Comparar tres métodos *forecast-valid* de imputación: drift naive y ARIMA con `sktime`, `IterativeImputer` + RF con `sklearn`.
2. Crear dos bloques de huecos en regímenes contrarios — **invierno estable** y **verano dinámico** — para ver cómo el "mejor método" depende del régimen.
3. Entender que más complejidad no garantiza mejor imputación: ARIMA brilla en series planas, ML brilla con dinámica y exógenas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sktime.transformations.series.impute import Imputer
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.arima import ARIMA

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(0)

## 1 · Datos y huecos artificiales

Partimos de la serie diaria limpia de Pinos-Genil 2011-2020 + lluvia de la cuenca como exógena (sin huecos artificiales, la usaremos como feature para el método ML).

Creamos **tres tipos de huecos** sobre una copia del caudal:

- **MCAR (10%).** Días aislados, dispersos — sensor con dropouts esporádicos.
- **Bloque invierno (45 días, Feb 2018).** Caudal bajo y estable, sin crecidas — escenario fácil para modelos de persistencia.
- **Bloque verano (45 días, Ago 2018).** Caudal medio con variabilidad por sueltas de embalse — escenario donde la dinámica importa.

In [ ]:
caudal = ud.cargar_caudal_genil(source="CEDEX")
lluvia = ud.cargar_lluvia_genil_diaria(fecha_inicio="2010-01-01", fecha_fin="2020-12-31")

df = pd.DataFrame({"caudal": caudal, "lluvia": lluvia}).loc["2011":"2020"]
df = df.asfreq("D").interpolate("linear", limit=7).dropna()

y_true = df["caudal"]
y_true.name = "caudal"
y_gap = y_true.copy()

# 1) MCAR 10%
mask_mcar = rng.random(len(y_gap)) < 0.10

# 2) Bloque invierno (Feb 2018) — flujo bajo y estable
block_w_start = pd.Timestamp("2018-02-01")
block_w_end = block_w_start + pd.Timedelta(days=44)
mask_winter = (y_gap.index >= block_w_start) & (y_gap.index <= block_w_end)

# 3) Bloque verano (Ago 2018) — caudal con variabilidad
block_s_start = pd.Timestamp("2018-08-01")
block_s_end = block_s_start + pd.Timedelta(days=44)
mask_summer = (y_gap.index >= block_s_start) & (y_gap.index <= block_s_end)

mask_missing = mask_mcar | mask_winter | mask_summer
y_gap[mask_missing] = np.nan

print(f"Total días: {len(y_gap):,}   Huecos totales: {y_gap.isna().sum():,}")
print(f"  MCAR:    {mask_mcar.sum():,}  ({mask_mcar.mean():.1%})")
print(f"  Invierno: {mask_winter.sum():,}  ({block_w_start.date()} → {block_w_end.date()})")
print(f"  Verano:   {mask_summer.sum():,}  ({block_s_start.date()} → {block_s_end.date()})")
print()
print(f"Truth invierno: mean={y_true[mask_winter].mean():.3f}  std={y_true[mask_winter].std():.3f}")
print(f"Truth verano:   mean={y_true[mask_summer].mean():.3f}  std={y_true[mask_summer].std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(y_true.index, y_true.values, color="#999", lw=0.6, label="verdadero")
ax.plot(y_gap.index, y_gap.values, color="black", lw=0.6, label="con huecos")
ax.axvspan(block_w_start, block_w_end, color="#fca5a5", alpha=0.4, label="bloque invierno")
ax.axvspan(block_s_start, block_s_end, color="#fcd34d", alpha=0.4, label="bloque verano")
ax.set_ylabel("Q (m³/s)")
ax.set_title("Serie con huecos artificiales")
ax.legend(ncol=4)
plt.tight_layout()

## 2 · Función de evaluación

Reportamos RMSE separado para MCAR, bloque invierno y bloque verano — porque vamos a ver que **cada método gana en un escenario distinto**.

In [ ]:
def evaluar(y_hat, nombre):
    def rmse(a, b):
        return float(np.sqrt(np.mean((a - b) ** 2)))

    return {
        "método": nombre,
        "RMSE_mcar": rmse(y_true[mask_mcar], y_hat[mask_mcar]),
        "RMSE_invierno": rmse(y_true[mask_winter], y_hat[mask_winter]),
        "RMSE_verano": rmse(y_true[mask_summer], y_hat[mask_summer]),
        "RMSE_total": rmse(y_true[mask_missing], y_hat[mask_missing]),
    }


resultados = []

## 3 · Forecast-valid: la única restricción

Un método de imputación es *forecast-valid* si para rellenar el día $t$ no usa ningún valor de $t' \geq t$. Sin esa propiedad, los huecos imputados contienen información del futuro y un modelo entrenado encima sobreestima su rendimiento real.

- **sktime.** `Imputer(method="forecaster", forecaster=...)` lo es por construcción. `method="linear"`, `"bfill"`, etc. **no**.
- **sklearn `IterativeImputer`.** Forecast-valid si la matriz de features solo contiene lags pasados (`shift(+k)`) y exógenas conocidas en $t$. Si añades `shift(-k)` o features que dependan del futuro, deja de serlo.

Los tres métodos del notebook respetan esta restricción.

## 4 · Método 1 — Drift naive

`NaiveForecaster(strategy=\"drift\")` ajusta una pendiente entre el primer y el último valor observado de la ventana de entrenamiento y la extrapola. Dentro del `Imputer`: para cada hueco, predice una recta desde el último día observado anterior.

Es el equivalente forecast-valid de la interpolación lineal y nuestro nuevo baseline.

In [ ]:
imp_drift = Imputer(method="forecaster", forecaster=NaiveForecaster(strategy="drift"))
y_drift = imp_drift.fit_transform(y_gap)

resultados.append(evaluar(y_drift, "Drift naive (forecast-valid)"))
print(resultados[-1])

## 5 · Método 2 — ARIMA

Modelo estadístico clásico vía sktime — `ARIMA` envuelve `statsmodels.SARIMAX`. Capta la dinámica autoregresiva corta; para huecos cortos predice muy cerca de la persistencia, para bloques largos converge a la media local de la diferencia.

In [ ]:
imp_arima = Imputer(
    method="forecaster",
    forecaster=ARIMA(order=(2, 1, 1), suppress_warnings=True),
)
y_arima = imp_arima.fit_transform(y_gap)

resultados.append(evaluar(y_arima, "ARIMA(2,1,1)"))
print(resultados[-1])

## 6 · Método 3 — `IterativeImputer` + RF (sklearn)

Para meter un modelo ML en la imputación, sktime no nos sirve: `make_reduction(rf, ...)` no soporta predicción in-sample (lanza `NotImplementedError`). La alternativa canónica es **sklearn `IterativeImputer`** — trata la imputación como un problema multivariante y rellena las columnas iterativamente con un regresor.

Matriz:

- `caudal` (target con NaN)
- Tres lags pasados: `q_lag1, q_lag7, q_lag365`
- Dos features Fourier: `sin_an, cos_an`
- Dos features de lluvia: `p_lag1, p_acum7` (lluvia de ayer y acumulada 7 días)

Solo lags pasados y exógenas conocidas en $t$ → forecast-valid.

In [ ]:
mat = pd.DataFrame({"caudal": y_gap})
for lag in (1, 7, 365):
    mat[f"q_lag{lag}"] = y_gap.shift(lag)
idx = mat.index
mat["sin_an"] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
mat["cos_an"] = np.cos(2 * np.pi * idx.dayofyear / 365.25)
mat["p_lag1"] = df["lluvia"].shift(1)
mat["p_acum7"] = df["lluvia"].rolling(7).sum().shift(1)
mat = mat.loc["2012":"2020"]  # recortar bordes donde lag_365 es NaN

imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0),
    max_iter=5,
    random_state=0,
)
mat_imp = pd.DataFrame(imputer.fit_transform(mat), index=mat.index, columns=mat.columns)

y_iter = y_gap.copy()
fill_mask = y_iter.isna() & y_iter.index.isin(mat_imp.index)
y_iter.loc[fill_mask] = mat_imp.loc[fill_mask, "caudal"]
y_iter = y_iter.interpolate("time")  # bordes pre-2012

resultados.append(evaluar(y_iter, "IterativeImputer + RF"))
print(resultados[-1])

## 7 · Comparativa

In [ ]:
tabla = pd.DataFrame(resultados).set_index("método").round(3)
print(tabla)

In [ ]:
def plot_bloque(ax, start, end, titulo, span_color):
    ventana = slice(start - pd.Timedelta(days=15), end + pd.Timedelta(days=15))
    ax.plot(
        y_true.loc[ventana].index,
        y_true.loc[ventana].values,
        color="black",
        lw=1.4,
        label="verdadero",
    )
    for nombre, serie, color in [
        ("drift naive", y_drift, "#2563eb"),
        ("ARIMA", y_arima, "#c2410c"),
        ("IterativeImputer + RF", y_iter, "#16a34a"),
    ]:
        ax.plot(
            serie.loc[ventana].index,
            serie.loc[ventana].values,
            color=color,
            ls="--",
            lw=1.0,
            label=nombre,
        )
    ax.axvspan(start, end, color=span_color, alpha=0.25)
    ax.set_ylabel("Q (m³/s)")
    ax.set_title(titulo)


fig, ax = plt.subplots(figsize=(11, 4))
plot_bloque(
    ax, block_w_start, block_w_end, "Bloque INVIERNO (Feb 2018) — flujo bajo y estable", "#fca5a5"
)
ax.legend(ncol=4, fontsize=9)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
plot_bloque(
    ax, block_s_start, block_s_end, "Bloque VERANO (Ago 2018) — caudal con variabilidad", "#fcd34d"
)
ax.legend(ncol=4, fontsize=9)
plt.tight_layout()

## 8 · Lectura — qué gana en cada régimen

La tabla y los dos zooms cuentan la misma historia:

- **MCAR.** Diferencias pequeñas — para huecos de 1 día, todos los métodos hacen prácticamente lo mismo: el último valor observado es casi exacto. ARIMA gana por márgenes mínimos.
- **Bloque invierno.** ARIMA y drift aplastan al ML (RMSE ~0.02 frente a ~0.5). La serie está esencialmente plana (`std=0.015`); persistir la media local es óptimo. El RF, entrenado sobre 10 años con muchas crecidas, no extrapola a este invierno particular y produce predicciones erráticas.
- **Bloque verano.** El IterativeImputer + RF gana claramente (RMSE ~0.17 vs ~0.38 de ARIMA y drift). Aquí hay dinámica: el caudal sube y baja por sueltas, la lluvia importa, y los lags + Fourier + lluvia le dan al RF señal que aprovechar. ARIMA, sin exógenas, solo puede extrapolar y se queda en la media.

**Moraleja: no hay método universal.** El "mejor imputador" depende del régimen del bloque. En series planas con buena autocorrelación, el modelo estadístico simple bate al ML. Cuando hay dinámica y exógenas con respuesta clara, el ML toma la delantera. Si no sabes en qué régimen está tu hueco, prueba varios y compara contra MCAR sintéticos en periodos similares.

## 9 · Ejercicios

1. **Más lags.** Añade `q_lag30, q_lag90` a la matriz del `IterativeImputer`. ¿Mejora el bloque verano? ¿Empeora el invierno?
2. **Quitar la lluvia.** Elimina `p_lag1, p_acum7` del IterativeImputer. ¿Se cae la ventaja en verano?
3. **Cambio de estimador.** Sustituye `RandomForestRegressor` por `BayesianRidge()` (el default). Compara RMSE y tiempo. El comportamiento errático en invierno, ¿es del RF o del esquema iterativo?
4. **AutoARIMA.** Reemplaza `ARIMA(...)` por `from sktime.forecasting.arima import AutoARIMA; AutoARIMA(suppress_warnings=True)`. ¿Encuentra un orden mejor? ¿Reduce el RMSE del bloque verano (donde ARIMA pierde)?
5. **Otros años.** Cambia `block_w_start` y `block_s_start` a 2014 o 2016. ¿Se mantiene el patrón "ARIMA gana plano, ML gana dinámico"?